# Ch 5 Discrimination Analysis

Fisher-exact discrimination of passage-level annotations across three axes:
- **`halfcent`** — period (all fiction), diachronic arc
- **`pair`** — genre × genre combinatorics (tag intersections across all time)
- **`halfcent_tag`** — period × genre (period-controlled genre effects)

Uses `largeliterarymodels.analysis` (task adapters) + `lltk.analysis.stats` (engine).

In [ ]:
import numpy as np
import pandas as pd
import lltk
from plotnine import (
    ggplot, aes, geom_point, geom_vline, geom_tile, geom_line, geom_hline,
    facet_wrap, scale_color_manual, scale_fill_gradient2, scale_size_continuous,
    coord_flip, theme_bw, theme, element_text, element_blank, element_line, labs,
    scale_x_continuous, scale_y_continuous, annotate,
)
from largeliterarymodels.analysis import (
    joint_feature_matrix,
    passage_groups,
    fisher_tests,
    bh_fdr,
)

def extract_year(s):
    return pd.Series(s).str.extract(r"(\d{4})")[0].astype(float)


## 1. Feature matrix + text metadata

In [ ]:
feats = joint_feature_matrix(
    tasks=["passage-content", "passage-form"],
    task_versions={"passage-content": 2, "passage-form": 1},
    source_agents={
        "passage-content": "qwen3.5-35b-a3b",
        "passage-form":    "claude-sonnet-4-6",
    },
)
print(f"Shape: {feats.shape}  index levels: {feats.index.names}")
feats.head(3)

In [ ]:
# Many-to-one join: passages → text-level metadata
text_ids = sorted(feats.index.get_level_values("_id").unique())
id_list = ",".join(f"'{i}'" for i in text_ids)
meta = lltk.db.conn.execute(
    f"SELECT _id, year, corpus, genre, genre_raw FROM lltk.texts FINAL WHERE _id IN ({id_list})"
).fetchdf()
meta["halfcent"] = (meta["year"] // 50 * 50).astype(str) + "-" + (meta["year"] // 50 * 50 + 49).astype(str)
print(f"{len(meta)} texts, year range {meta['year'].min()}–{meta['year'].max()}")
meta.head(3)

## 2. Groups + Fisher tests

In [ ]:
# groups: bool DataFrame (passage × group_label)
# kind:   dict {group_label -> 'single'|'pair'|'halfcent'|'halfcent_tag'}
groups, kind = passage_groups(
    feats.index,
    include_halfcent=True,
    include_pairs=True,
)
print(f"{groups.shape[1]} groups — " + ", ".join(f"{k}: {sum(v==k for v in kind.values())}" for k in sorted(set(kind.values()))))

In [ ]:
results = fisher_tests(feats, groups)
results["q_value"] = bh_fdr(results["p_value"])
results["group_kind"] = results["group"].map(lambda g: kind.get(g, "feature"))
results["log_odds"] = np.log2(results["odds_ratio"].clip(0.01, 100))

sig = results[results["q_value"] < 0.001].sort_values("p_value")
print(f"Total: {len(results)} tests  |  Sig q<0.001: {len(sig)}")

## 3. Panel A — Diachronic arc (`halfcent`)

What is distinctive about each half-century across **all fiction**.

In [ ]:
sig_hc = sig[sig["group_kind"] == "halfcent"].copy()

# Top 30 by p-value; extract start year for ordering
top_hc = (
    sig_hc.nsmallest(30, "p_value")
    .assign(year=lambda d: d["group"].str.extract(r"(\d{4})").astype(float))
)

# Order features by mean log-odds for readability
feat_order = top_hc.groupby("feature")["log_odds"].mean().sort_values().index.tolist()
top_hc["feature"] = pd.Categorical(top_hc["feature"], categories=feat_order, ordered=True)

(
    ggplot(top_hc, aes(x="log_odds", y="feature", color="factor(year)", size="-log_odds".replace("-", "")))
    + geom_vline(xintercept=0, linetype="dashed", color="#cccccc")
    + geom_point(aes(size=np.abs(top_hc["log_odds"])))
    + labs(
        x="Log₂ odds ratio", y="", color="Half-cent.",
        title="Panel A: Diachronic arc — top 30 discriminating features (q < 0.001)"
    )
    + theme_bw()
    + theme(figure_size=(11, 9), axis_text_y=element_text(size=8), legend_position="right")
)

In [ ]:
# Heatmap: all sig halfcent features × period
feat_counts = sig_hc.groupby("feature")["group"].count()
keep_feats = feat_counts[feat_counts >= 3].index
heat = sig_hc[sig_hc["feature"].isin(keep_feats)].copy()

# Order features by peak period
feat_peak = heat.loc[heat.groupby("feature")["rate_in_group"].idxmax(), ["feature", "group"]]
feat_peak = feat_peak.sort_values("group")
heat["feature"] = pd.Categorical(heat["feature"], categories=feat_peak["feature"].tolist(), ordered=True)

(
    ggplot(heat, aes(x="group", y="feature", fill="rate_in_group"))
    + geom_tile()
    + scale_fill_gradient2(
        low="#2166ac", mid="#f7f7f7", high="#d6604d",
        midpoint=heat["rate_in_group"].median()
    )
    + labs(
        x="Half-century", y="", fill="Rate",
        title="Feature rate by half-century — ordered by peak period"
    )
    + theme_bw()
    + theme(
        figure_size=(14, max(6, len(keep_feats) * 0.25)),
        axis_text_x=element_text(angle=45, ha="right", size=8),
        axis_text_y=element_text(size=7),
        panel_grid=element_blank()
    )
)

## 4. `concrete_bespeaks_abstract` — the cliff (key Ch5 visualization)

All-fiction baseline (`halfcent`) vs novel-only (`halfcent_tag`) on the same axes.

In [ ]:
cba_hc = (
    results[
        results["feature"].str.contains("concrete_bespeaks_abstract", case=False, na=False)
        & (results["group_kind"] == "halfcent")
    ]
    .assign(year=lambda d: extract_year(d["group"]), series="All fiction")
)

cba_ht = (
    results[
        results["feature"].str.contains("concrete_bespeaks_abstract", case=False, na=False)
        & (results["group_kind"] == "halfcent_tag")
        & results["group"].str.contains("novel", case=False, na=False)
    ]
    .assign(year=lambda d: extract_year(d["group"]), series="Novel only")
)

cba = pd.concat([cba_hc, cba_ht], ignore_index=True)
print(cba[["series", "group", "year", "rate_in_group", "rate_not_group", "odds_ratio", "p_value"]]
      .sort_values(["series", "year"]).to_string(index=False))

In [ ]:
(
    ggplot(cba[cba["year"].notna()], aes(x="year", y="rate_in_group", color="series", linetype="series"))
    + geom_line(size=1.2)
    + geom_point(aes(size=-np.log10(cba[cba["year"].notna()]["p_value"].clip(1e-100))))
    + geom_hline(yintercept=cba_hc["rate_not_group"].mean(), linetype="dotted", color="#888888", size=0.8)
    + scale_color_manual(values={"All fiction": "#2166ac", "Novel only": "#d6604d"})
    + scale_size_continuous(range=(1, 5), guide=None)
    + labs(
        x="Half-century start", y="Rate of concrete_bespeaks_abstract",
        color="", linetype="",
        title="concrete_bespeaks_abstract: all fiction vs novel-only",
        caption="Point size ∝ –log₁₀(p).  Dotted line = baseline rate outside group."
    )
    + theme_bw()
    + theme(figure_size=(9, 5), legend_position="bottom")
)

## 5. Panel B — Genre combinatorics (`pair`)

Tag × tag intersections across all time — which genre co-occurrences are over/under-represented.

In [ ]:
sig_pair = sig[sig["group_kind"] == "pair"].sort_values("p_value")
print(f"{len(sig_pair)} sig pair groups")

top_pair = sig_pair.nsmallest(25, "p_value")
feat_order_p = top_pair.groupby("feature")["log_odds"].mean().sort_values().index.tolist()
top_pair = top_pair.copy()
top_pair["feature"] = pd.Categorical(top_pair["feature"], categories=feat_order_p, ordered=True)

(
    ggplot(top_pair, aes(x="log_odds", y="feature", color="group"))
    + geom_vline(xintercept=0, linetype="dashed", color="#cccccc")
    + geom_point(size=3)
    + labs(
        x="Log₂ odds ratio", y="", color="Genre pair",
        title="Panel B: Genre combinatorics — top 25 pair contrasts (q < 0.001)"
    )
    + theme_bw()
    + theme(figure_size=(11, 8), axis_text_y=element_text(size=8), legend_text=element_text(size=7))
)

## 6. Panel C — Period-controlled genre (`halfcent_tag`)

Genre effects *within* each half-century — controls for composition shift confound.

In [ ]:
sig_ht = sig[sig["group_kind"] == "halfcent_tag"].copy()
sig_ht["year"] = extract_year(sig_ht["group"]).values
sig_ht["tag"] = sig_ht["group"].str.replace(r"\d{4}-\d{4}\|", "", regex=True)
print(f"{len(sig_ht)} sig halfcent_tag results  |  tags: {sorted(sig_ht['tag'].unique())}")

In [ ]:
# Top features per tag — bubble chart: x=year, y=feature, size=|log_odds|, color=tag
top_ht = sig_ht.nsmallest(40, "p_value")
feat_order_ht = top_ht.groupby("feature")["log_odds"].mean().sort_values().index.tolist()
top_ht["feature"] = pd.Categorical(top_ht["feature"], categories=feat_order_ht, ordered=True)

(
    ggplot(top_ht, aes(x="year", y="feature", color="tag", size=np.abs(top_ht["log_odds"])))
    + geom_point(alpha=0.8)
    + scale_size_continuous(range=(2, 8), guide=None)
    + labs(
        x="Half-century start", y="", color="Genre tag",
        title="Panel C: Period-controlled genre — top 40 halfcent×tag contrasts",
        caption="Size ∝ |log₂ odds|"
    )
    + theme_bw()
    + theme(figure_size=(11, 9), axis_text_y=element_text(size=8))
)

## 7. Content V3 — headline findings

V3 has 5,844 passages (826 texts, 1450–1800). Single-task for max statistical power.

In [ ]:
feats_v3 = joint_feature_matrix(
    tasks=["passage-content"],
    task_versions={"passage-content": 3},
    source_agents={"passage-content": "qwen3.5-35b-a3b"},
)
groups_v3, kind_v3 = passage_groups(
    feats_v3.index, include_halfcent=True, include_pairs=True
)
res_v3 = fisher_tests(feats_v3, groups_v3)
res_v3["q_value"] = bh_fdr(res_v3["p_value"])
res_v3["group_kind"] = res_v3["group"].map(lambda g: kind_v3.get(g, "feature"))
res_v3["log_odds"] = np.log2(res_v3["odds_ratio"].clip(0.01, 100))

sig_v3 = res_v3[res_v3["q_value"] < 0.001].sort_values("p_value")
print(f"V3: {len(sig_v3)} sig at q<0.001 / {len(res_v3)} total")
sig_v3.head(20)

In [ ]:
# Key Ch5 features from V3
headline_feats = ["gentry_or_middling", "domestic_routine", "chivalric", "concrete_bespeaks_abstract"]
headline = (
    sig_v3[
        sig_v3["feature"].str.contains("|".join(headline_feats), case=False, na=False)
        & sig_v3["group_kind"].isin(["halfcent", "halfcent_tag"])
    ]
    .sort_values("p_value")
)
print(headline[["feature", "group", "group_kind", "rate_in_group", "odds_ratio", "p_value", "q_value"]]
      .head(20).to_string(index=False))